# Clase 041 — Seaborn

**Parte 0** · VanderPlas cap. 4 § 4.13.

> 🎯 matplotlib + defaults + API tipada para DataFrames. Pairplot, hue, facetas.

> ⏱️ ~75 min

## ⚙️ Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='deep')

try:
    peng = sns.load_dataset('penguins').dropna()
    print(f'penguins cargado: {peng.shape}')
except Exception as e:
    print(f'fallback sintético: {e}')
    rng = np.random.default_rng(42)
    peng = pd.DataFrame({
        'species': np.repeat(['Adelie', 'Chinstrap', 'Gentoo'], [50, 30, 40]),
        'sex'    : np.tile(['Male', 'Female'], 60),
        'bill_length_mm'  : np.concatenate([rng.normal(39, 2, 50), rng.normal(48, 3, 30), rng.normal(48, 3, 40)]),
        'bill_depth_mm'   : np.concatenate([rng.normal(18, 1, 50), rng.normal(18, 1, 30), rng.normal(15, 1, 40)]),
        'flipper_length_mm': np.concatenate([rng.normal(190, 6, 50), rng.normal(196, 7, 30), rng.normal(217, 7, 40)]),
        'body_mass_g'     : np.concatenate([rng.normal(3700, 400, 50), rng.normal(3700, 400, 30), rng.normal(5050, 500, 40)]),
    })

## 1️⃣ Pairplot — EDA en una línea

In [ ]:
g = sns.pairplot(peng, hue='species', diag_kind='kde', height=2)
plt.show()

## 2️⃣ Figure-level vs axes-level

| Categoría | Figure-level | Axes-level |
|---|---|---|
| Distribuciones | `displot` | `histplot`, `kdeplot`, `ecdfplot` |
| Relaciones | `relplot` | `scatterplot`, `lineplot` |
| Categóricas | `catplot` | `boxplot`, `violinplot`, `stripplot`, `swarmplot` |

**Figure-level**: hace su propia figure, soporta facetas (`col`, `row`).  
**Axes-level**: dibuja en un `ax` que tú le pases — integra con grids matplotlib custom.

Regla simple: si quieres facetas, figure-level. Si necesitas control fino del layout, axes-level.

## 3️⃣ Scatter con hue, style, size

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(
    data=peng, x='bill_length_mm', y='body_mass_g',
    hue='species', style='sex', size='flipper_length_mm',
    sizes=(20, 200), alpha=0.7, ax=ax,
)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 4️⃣ Distribuciones

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=peng, x='body_mass_g', hue='species', kde=True, ax=a1)
a1.set_title('histplot + KDE')
sns.violinplot(data=peng, x='species', y='body_mass_g', ax=a2)
a2.set_title('violin')
plt.tight_layout()
plt.show()

## 5️⃣ Boxplot + swarmplot combinado

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=peng, x='species', y='body_mass_g', ax=ax, color='lightgray')
sns.swarmplot(data=peng, x='species', y='body_mass_g', hue='sex', ax=ax, size=4)
plt.title('box + swarm (mejor de ambos mundos)')
plt.tight_layout()
plt.show()

## 6️⃣ Facetas — figure-level

In [ ]:
g = sns.relplot(
    data=peng, x='bill_length_mm', y='body_mass_g',
    hue='sex', col='species', kind='scatter', height=4, aspect=1,
)
g.set_titles('{col_name}')
plt.show()

## 7️⃣ Themes y paletas

```python
sns.set_theme(
    style='whitegrid',          # darkgrid, white, dark, ticks
    palette='deep',             # muted, bright, pastel, dark, colorblind, husl
    font_scale=1.0,
)
```

Un solo `set_theme` afecta todos los plots de la sesión (incluso matplotlib puro).

## ✅ Checklist

- [ ] Uso pairplot para EDA inicial
- [ ] Distingo figure-level (facetas) vs axes-level (control fino)
- [ ] Codifico con hue/style/size
- [ ] Hago facetas con col=/row=
- [ ] Configuro tema global con set_theme

## 📝 Homework

Ver `README.md`. Pairplot, violin+swarm, facetas 2×3, tema custom.

## 📖 Definiciones y características

**Seaborn**

Wrapper sobre matplotlib con (1) defaults estéticos mejores, (2) API tipada para DataFrames (`x=`, `y=`, `hue=`), (3) plots estadísticos directos (regresión, distribuciones, facetas).

**Figure-level (displot, relplot, catplot)**

Funciones que crean su propia figura, soportan **facetas** (`col=`, `row=` para grilla automática). Más alto nivel, menos control fino.

**Axes-level (histplot, scatterplot, boxplot)**

Funciones que dibujan en un `ax` que tú pasas. Más bajo nivel, integran con layouts custom de matplotlib.

**`hue`, `style`, `size`**

Codifican dimensiones extra: **`hue`** (color por categoría), **`style`** (marker por categoría), **`size`** (tamaño por valor continuo).

**`pairplot`**

Matriz de scatterplots de todas las parejas de variables numéricas, diagonal con KDE/histograma. EDA visual en una línea.

**Faceta (col/row)**

Una sub-figura por cada valor de una categórica. `relplot(..., col='species', row='sex')` produce grilla `n_species × n_sex` de plots automática.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| Mezclo `sns.set_theme()` con `plt.style.use(...)` | Compiten por los rcParams. **Fix**: elige uno. Si usas seaborn, `sns.set_theme(style='whitegrid')` cubre todo. |
| Figure-level no me deja agregar `ax.set_title()` | Devuelve `FacetGrid`, no axes. **Fix**: `g.set_titles('{col_name}')` o `g.fig.suptitle(...)` para título global; `g.axes_dict` para acceder a subplots. |
| `pairplot` con muchas columnas tarda eternidades | N² scatters; con 20 cols son 400 plots. **Fix**: selecciona subset relevante (`peng[['masa','pico','aleta','species']]`) o usa `corr()` heatmap. |
| `hue` con muchas categorías → leyenda enorme | Default muestra todas las categorías. **Fix**: limita con `hue_order=[5 más relevantes]`, o agrupa las menores en "otros". |
| `sns.scatterplot(...)` no aparece en notebook | Falta capturar return o `plt.show()`. **Fix**: asignar a `ax = sns.scatterplot(...)` o llamar `plt.show()` al final. |

## ❓ Preguntas frecuentes

**❓ ¿`seaborn` o `matplotlib` puro?**

**Seaborn** para plots estadísticos rápidos con DataFrame (`hue`, facetas, defaults). **Matplotlib** para control fino, customización extrema, o plots que no son estadísticos.

**❓ ¿Cuándo figure-level vs axes-level?**

**Figure-level** si quieres facetas o el plot es el output completo. **Axes-level** si necesitas integrar con layout custom (subplots manuales).

**❓ ¿`pairplot` o `corrmatrix`?**

**`pairplot`** muestra relaciones visualmente (no lineales, outliers, clusters). **Heatmap de `corr()`** muestra coeficiente Pearson (solo lineal). Complementarios.

**❓ ¿Tema corporativo en seaborn?**

`sns.set_theme(palette=['#0F766E', '#D9A441', '#7C3AED'])` o palette custom: `sns.color_palette('husl', n_colors=5)`. Combina con `style='whitegrid'`.

**❓ ¿Seaborn maneja datasets grandes (>1M)?**

Plots scatter/hist sí. Pairplot con muchas cols se vuelve lento. Para datasets enormes, considera datashader o downsample antes.

## 🔗 Referencias

- VanderPlas cap. 4 § 4.13
- [seaborn tutorial](https://seaborn.pydata.org/tutorial.html)

➡️ **Siguiente:** [040 — Visualización geográfica](../040-visualizacion-geografica-plotly-folium/README.md)